In [6]:
import os
import sys
import subprocess
import pandas as pd
import numpy as np

In [7]:
TRAIN_FILE = "train.csv"
TEST_FILE = "test.csv"
import pandas as pd

train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv", usecols=['TEXT'])
print(train_df.columns)
print(test_df.columns)
print(train_df.head())
print(test_df.head())

Index(['TEXT', 'LABEL'], dtype='str')
Index(['TEXT'], dtype='str')
                                                TEXT  LABEL
0  Dear state senator, There should be a change i...      0
1  A star's life cycle begins in a nebula and pro...      1
2  Limiting the usage of has a variety advantages...      0
3  Biodiversity loss accelerates at unprecedented...      3
4  The morning had been cold. The hours dawn stil...      0
                                                TEXT
0  The Elecctor College ishould be aboliished bea...
1  I WAS ESPECIALLY DELIGHTED THAT IN THIS MOVIE ...
2  `` ONE BRAIN,''' I scream down from the rtee. ...
3  A day without driving your car? who can't do t...
4  If you were told on some days of the wek that ...


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer


vectorizer = TfidfVectorizer(
    #max_features=100000,
    max_features=50_000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    max_df=0.95,
    strip_accents='unicode',
    lowercase=True,
    analyzer='word',
    token_pattern=r'\b[a-zA-Z]+\b',
    # stop_words='english' 
)


In [9]:
train_tfidf = vectorizer.fit_transform(train_df['TEXT'])
#print(train_tfidf)

test_tfidf = vectorizer.transform(test_df['TEXT'])


feature_names = vectorizer.get_feature_names_out()
#print(feature_names)

#term_scores1 = train_tfidf.sum(axis=0)
term_scores = train_tfidf.sum(axis=0).A1 
#print(term_scores)
#print(term_scores1)

#top_idx = term_scores.argsort()[::-1][:20]
#print(top_idx)
#for i, idx in enumerate(top_idx, 1):
    #print(f"{i:2d}. {feature_names[idx]}")

In [10]:
from sklearn.linear_model import LogisticRegression



model = LogisticRegression(max_iter=1000,
random_state=42, 
class_weight='balanced')

y_train = train_df['LABEL']
model.fit(train_tfidf, y_train)
y_test = model.predict(test_tfidf)
test_df['PREDICTED_RESULT'] = y_test
submission = pd.DataFrame({
    'ID': test_df.index,
    'LABEL': test_df['PREDICTED_RESULT']
})
submission.to_csv('submission.csv', index=False)

## Note

Although I know the theoretical concepts of the following solution with BERT. I don't have enough experience to implement it; therefore AI tools were used to generate the following parts and the score was improved to 0.9 from 0.86. However, the scores generated by the following BERT solution were not selected on Kaggle. Only the scores generated by the Logistic Regression solution were chosen on Kaggle.

In [1]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        enc = self.tokenizer(
            self.texts[i],
            padding="max_length",
            max_length=self.max_length,
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[i], dtype=torch.long),
        }

MODEL_BERT_DIR = "model_bert"
BASE_MODEL = "distilbert-base-uncased"
MAX_LEN = 256
EPOCHS = 4
BATCH = 8
LR = 2e-5
USE_CPU = True 


/Users/suleymanbdaziroglu/Desktop/malto-hackathon/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
texts = train_df["TEXT"].astype(str).fillna("").tolist()
labels = train_df["LABEL"].astype(int).tolist()

from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=6)
if USE_CPU:
    model = model.to("cpu")

dataset = TextDataset(texts, labels, tokenizer, MAX_LEN)

training_args = TrainingArguments(
    output_dir=MODEL_BERT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    learning_rate=LR,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    dataloader_pin_memory=False,
)

trainer = Trainer(model=model, args=training_args, train_dataset=dataset)
trainer.train()

out = trainer.predict(dataset)
preds = out.predictions.argmax(axis=1)


trainer.save_model(MODEL_BERT_DIR)
tokenizer.save_pretrained(MODEL_BERT_DIR)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3191.60it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
50,1.020952
100,0.346497
150,0.193863
200,0.166224
250,0.109257
300,0.122245
350,0.102184
400,0.089604
450,0.088874
500,0.063814


Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.75s/it]


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.54s/it]


('model_bert/tokenizer_config.json', 'model_bert/tokenizer.json')

In [12]:
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification


model = AutoModelForSequenceClassification.from_pretrained(MODEL_BERT_DIR)
tokenizer = AutoTokenizer.from_pretrained(MODEL_BERT_DIR)
model.eval()
device = torch.device("cpu")  # M1 bellek sorunu yaşamamak için CPU
model.to(device)

test_texts = test_df["TEXT"].astype(str).fillna("").tolist()
all_preds = []
bs = 16
for i in range(0, len(test_texts), bs):
    batch = test_texts[i:i+bs]
    enc = tokenizer(batch, padding=True, max_length=MAX_LEN, truncation=True, return_tensors="pt")
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits
        all_preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())

submission_bert = pd.DataFrame({"ID": range(len(all_preds)), "LABEL": all_preds})
submission_bert.to_csv("submission_bert.csv", index=False)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1427.61it/s]
